# NER eğitimi ve değerlendirme (kişi / kurum / yer / adres)

- **Kod** her çalıştırmada GitHub'dan (`irmakyaprakk/turkce-llm-dlp`, `main`) alınır.
- **Veri** Drive'daki `turkce-dlp-colab-bundle-*` paketlerinden gelir (zip ya da klasör; adı önemsiz).
- **Model ve sonuçlar** Drive'da `turkce-llm-dlp/runs/<deney>/` altına kaydedilir. Aynı deney
  için model zaten varsa eğitim atlanır, yalnızca (güncel kodla) yeniden değerlendirilir.

Çalıştırma: Çalışma zamanı → **T4 GPU** → ayar hücresinde deneyi seç → **Tümünü çalıştır**.

In [ ]:
# 0) Deney ayarı — yalnızca burayı değiştir
DENEY = 'augmented_v2'   # 'baseline' | 'augmented' | 'augmented_v2'
SEED = 42

DATA_DIR = {'baseline': 'data/ner/processed/real-address-context-v1',
            'augmented': 'data/ner/processed/augmented-v1',
            'augmented_v2': 'data/ner/processed/augmented-v2'}[DENEY]
RUN = {'baseline': f'baseline_ctx_seed{SEED}', 'augmented': f'augmented_seed{SEED}',
       'augmented_v2': f'augmented_v2_seed{SEED}'}[DENEY]
print('Deney:', RUN, '| veri:', DATA_DIR)

In [ ]:
# 1) GPU ve Drive
import torch
assert torch.cuda.is_available(), "GPU yok: Çalışma zamanı türünü T4 GPU yap ve yeniden başlat."
print(torch.cuda.get_device_name(0))
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) Kod GitHub'dan, veri Drive'daki paket(ler)den
import os, pathlib, shutil, zipfile, subprocess
DRIVE = pathlib.Path('/content/drive/MyDrive/turkce-llm-dlp')
DRIVE.mkdir(parents=True, exist_ok=True)
os.chdir('/content')  # yeniden çalıştırmada silinecek klasörün içinde olmayalım
shutil.rmtree('/content/proje', ignore_errors=True)
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/irmakyaprakk/turkce-llm-dlp.git', '/content/proje'], check=True)
!cd /content/proje && git log --oneline -1

def add_data(source):
    # Paketteki yalnızca data/ klasörünü projeye ekler (kod GitHub'dan gelir).
    tmp = pathlib.Path('/content/_paket'); shutil.rmtree(tmp, ignore_errors=True)
    if source.is_dir():
        shutil.copytree(source, tmp)
    else:
        with zipfile.ZipFile(source) as zf: zf.extractall(tmp)
    root = next((p.parent for p in tmp.rglob('BUNDLE_INFO.txt')), tmp)
    if (root / 'data').exists():
        shutil.copytree(root / 'data', '/content/proje/data', dirs_exist_ok=True)
    print('Veri eklendi:', source.name, '-', (root / 'BUNDLE_INFO.txt').read_text().strip() if (root / 'BUNDLE_INFO.txt').exists() else '')

sources = {info.parent for info in pathlib.Path('/content/drive/MyDrive').rglob('BUNDLE_INFO.txt')}
sources |= set(pathlib.Path('/content/drive/MyDrive').rglob('turkce-dlp-colab-bundle*.zip'))
for source in sorted(sources, key=lambda p: p.name):   # v1 → v2 → v3; yenisi eskisinin üzerine yazar
    add_data(source)
%cd /content/proje
# Altın set GitHub'daki elle yazılmış partilerden üretilir (paket gerekmez).
!python training/ner/build_gold_set.py --out-dir data/ner/processed/realistic-v2 > /dev/null && echo 'Altın set hazır'
for need in (DATA_DIR, 'data/ner/processed/realistic-v2/test.jsonl', 'data/ner/processed/openstreetmap-tr/sentence-context-v1/test.jsonl'):
    assert pathlib.Path(need).exists(), f'{need} yok — gerekli veri paketini Drive\'a yükle'

In [ ]:
# 3) Bağımlılıklar ve veri doğrulama
!pip -q install -r training/ner/requirements.txt
!USE_TF=0 python training/ner/train.py --data-dir {DATA_DIR} --labels-path training/ner/labels.json --validate-only > /dev/null && echo 'Veri geçerli'

In [ ]:
# 4) Eğitim (Drive'da model varsa atlanır). T4'te yaklaşık 45-75 dk.
OUT = f'/content/runs/{RUN}'
saved = DRIVE / 'runs' / RUN
if (saved / 'model').exists():
    print('Drive\'da hazır model bulundu, eğitim atlanıyor:', saved)
    shutil.rmtree(OUT, ignore_errors=True)
    shutil.copytree(saved, OUT)
else:
    !USE_TF=0 python training/ner/train.py --data-dir {DATA_DIR} --labels-path training/ner/labels.json --output-dir {OUT} --epochs 3 --seed {SEED} --fp16
    shutil.rmtree(saved, ignore_errors=True)
    shutil.copytree(OUT, saved, ignore=shutil.ignore_patterns('checkpoints'))
    print('Drive\'a kaydedildi:', saved)

In [ ]:
# 5) Ek testler: cümle içi OSM adresleri + ALTIN TEST SETİ
for name, data in (('context', 'data/ner/processed/openstreetmap-tr/sentence-context-v1'),
                   ('gold', 'data/ner/processed/realistic-v2')):
    !USE_TF=0 python training/ner/evaluate_model.py --model-dir {OUT}/model --data-dir {data} --split test --output-json {OUT}/{name}-test-metrics.json > /dev/null
    shutil.copy(f'{OUT}/{name}-test-metrics.json', saved / f'{name}-test-metrics.json')
print('Değerlendirme tamam')

In [ ]:
# 6) Bana yapıştırılacak özet (yalnızca sayılar, ham metin yok)
import json
def pick(d, keys=('precision', 'recall', 'f1', 'coverage')):
    return {k: round(v, 4) for k, v in d.items() if isinstance(v, (int, float)) and any(x in k for x in keys)}
summary = {'run': RUN}
m = json.load(open(f'{OUT}/metrics.json'))
for part in ('test_wiki', 'test_openstreetmap_real_addresses'):
    if part in m:
        summary[part] = {k.replace(part + '_', ''): v for k, v in pick(m[part]).items() if 'overall' in k or 'address' in k or part == 'test_wiki'}
for name in ('context', 'gold'):
    d = json.load(open(f'{OUT}/{name}-test-metrics.json'))
    summary[name] = {'katı': pick(d['metrics']), 'son_işlemli': pick(d.get('metrics_postprocessed', d.get('metrics_merged_adjacent', {}))),
                     'kapsama': pick(d.get('token_coverage', {})), 'destek': d.get('support')}
    for batch, b in d.get('by_batch', {}).items():
        summary[f'{name}:{batch}'] = {'satır': b['rows'], 'katı_f1': pick(b['metrics'], ('f1',)),
                                       'son_işlemli_f1': pick(b['metrics_postprocessed'], ('f1',)),
                                       'kapsama': pick(b['token_coverage'])}
print(json.dumps(summary, ensure_ascii=False, indent=1))